# Controlador Proporcional

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import ipywidgets as widgets
from IPython.display import display, clear_output

# Vector de tiempo
t = np.linspace(0, 12, 500)

# Slider
kp_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=15.0,
    step=0.1,
    description='kp:',
    continuous_update=False,
    layout=widgets.Layout(width='500px')
)

output = widgets.Output()


def simulate(change=None):

    kp = kp_slider.value

    # -------------------------------------------------
    # Sistema de lazo cerrado
    #
    #          kp
    # T(s) = --------------------
    #        s² + 1.4s + 1 + kp
    # -------------------------------------------------

    system = signal.TransferFunction(
        [kp],
        [1, 1.4, 1 + kp]
    )

    # Respuesta al escalón
    tout, y = signal.step(system, T=t)

    # Polos
    poles = np.roots([1, 1.4, 1 + kp])

    # Valor final
    y_final = kp / (1 + kp)

    # Error de estado estacionario
    ess = 1 / (1 + kp)

    # Sobrepaso
    ymax = np.max(y)

    overshoot = max(
        0,
        (ymax - y_final) / y_final * 100
    )

    # Tiempo de establecimiento (2 %)
    tolerance = 0.02 * y_final

    outside = np.where(
        np.abs(y - y_final) > tolerance
    )[0]

    if len(outside) == 0:
        ts = 0
    elif outside[-1] < len(t) - 1:
        ts = t[outside[-1] + 1]
    else:
        ts = np.nan


    # -------------------------------------------------
    # Mostrar resultados
    # -------------------------------------------------

    with output:

        clear_output(wait=True)

        fig, ax = plt.subplots(1, 2, figsize=(11, 4))

        # Respuesta temporal
        ax[0].plot(tout, y, label='Salida y(t)')
        ax[0].axhline(
            1,
            linestyle='--',
            label='Referencia'
        )
        ax[0].axhline(
            y_final,
            linestyle=':',
            label='Valor final'
        )

        ax[0].set_xlabel('Tiempo [s]')
        ax[0].set_ylabel('Salida')
        ax[0].set_title('Respuesta al escalón')
        ax[0].grid()
        ax[0].legend()

        # Polos
        ax[1].scatter(
            poles.real,
            poles.imag,
            marker='x',
            s=100
        )

        ax[1].axhline(0)
        ax[1].axvline(0)

        ax[1].set_xlim(-2, 0.5)
        ax[1].set_ylim(-4.5, 4.5)

        ax[1].set_xlabel('Parte real')
        ax[1].set_ylabel('Parte imaginaria')
        ax[1].set_title('Polos')
        ax[1].grid()

        plt.tight_layout()
        plt.show()

        # Resultados numéricos
        print(f"kp = {kp:.2f}")
        print(f"Valor final     = {y_final:.3f}")
        print(f"Error e_ss      = {ess:.3f}")
        print(f"Sobrepaso       = {overshoot:.1f} %")

        if np.isnan(ts):
            print("T. establecimiento > 12 s")
        else:
            print(f"T. establecimiento = {ts:.2f} s")


# Ejecutar cuando cambia el slider
kp_slider.observe(simulate, names='value')

display(kp_slider)
display(output)

simulate()

FloatSlider(value=1.0, continuous_update=False, description='kp:', layout=Layout(width='500px'), max=15.0, min…

Output()